# Golf Ball Detection & Tracking (Faster R-CNN + Kalman) di Google Colab

Runtime: **Runtime > Change runtime type > GPU (T4)**.

Notebook ini memakai kode yang sudah dipatch untuk PyTorch 2.x (`torchvision.ops` menggantikan ekstensi CUDA lama, jadi **tidak ada compile**).
Pastikan patch sudah di-push ke GitHub, atau ganti `REPO_URL` ke repo yang berisi patch.

In [ ]:
import torch
assert torch.cuda.is_available(), "Aktifkan GPU: Runtime > Change runtime type > GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Clone repo + dependency

In [ ]:
REPO_URL = "https://github.com/tegaryas/golf_ball.git"

%cd /content
!rm -rf golf_ball && git clone -q {REPO_URL}
%cd /content/golf_ball
!pip install -q easydict tensorboardX

## 2. Dataset

Taruh dataset di Google Drive, salah satu bentuk:
- `MyDrive/Golf Ball.zip` (lebih cepat, disarankan), atau
- folder `MyDrive/Golf Ball/` (berisi `Detection/` dan `Tracking/`).

Dataset disalin ke disk lokal Colab (`/content/dataset`). Drive asli tidak disentuh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "/content/drive/MyDrive"
if os.path.exists(f"{DRIVE}/Golf Ball.zip"):
    !mkdir -p /content/dataset && unzip -q -o "{DRIVE}/Golf Ball.zip" -d /content/dataset
else:
    !mkdir -p /content/dataset && cp -r "{DRIVE}/Golf Ball" /content/dataset/
DATA = "/content/dataset/Golf Ball"
assert os.path.isdir(f"{DATA}/Detection/JPEGImages"), f"Dataset tidak ketemu di {DATA}"
print(os.listdir(DATA))

In [ ]:
# Loader mengharapkan data/VOCdevkit2007/VOC2007. Detection punya train.txt/test.txt (cocok dengan voc_2007_train / voc_2007_test).
%cd /content/golf_ball
!mkdir -p data/VOCdevkit2007 data/pretrained_model
!rm -rf data/VOCdevkit2007/VOC2007 data/cache
!ln -s "{DATA}/Detection" data/VOCdevkit2007/VOC2007
!ls data/VOCdevkit2007/VOC2007/ImageSets/Main

## 3. Pretrained backbone (ResNet-101 Caffe, ~170 MB)

Mirror Zenodo; link Dropbox asli sudah mati.

In [ ]:
!wget -q -O data/pretrained_model/resnet101_caffe.pth https://zenodo.org/records/4513879/files/resnet101_caffe.pth
!md5sum data/pretrained_model/resnet101_caffe.pth   # harus 508530030bf473c7a976e922d69abaef

## 4. Training

Checkpoint **hanya disimpan di akhir tiap epoch** (1 epoch = 11.030 iterasi, bs=1). Disimpan langsung ke Drive supaya tidak hilang kalau sesi putus.
Nama file: `faster_rcnn_<session>_<epoch>_11029.pth`.

In [ ]:
EPOCHS = 10
SAVE_DIR = "/content/drive/MyDrive/golf_ball_models"

!python trainval_net.py --dataset pascal_voc --net res101 --cuda --bs 1 --nw 2 \
    --epochs {EPOCHS} --disp_interval 200 --save_dir {SAVE_DIR}

### Resume (kalau sesi putus)

Ganti `LAST_EPOCH` dengan epoch terakhir yang tersimpan di Drive, lalu jalankan.

In [ ]:
LAST_EPOCH = 1   # epoch terakhir yang ada filenya

!python trainval_net.py --dataset pascal_voc --net res101 --cuda --bs 1 --nw 2 \
    --epochs {EPOCHS} --disp_interval 200 --save_dir {SAVE_DIR} \
    --r True --checksession 1 --checkepoch {LAST_EPOCH} --checkpoint 11029

## 5. Demo: deteksi di gambar test

In [ ]:
CKPT_EPOCH = EPOCHS   # epoch yang mau dipakai
!rm -rf /content/demo_imgs && mkdir -p /content/demo_imgs
!head -n 6 data/VOCdevkit2007/VOC2007/ImageSets/Main/test.txt | while read n; do cp data/VOCdevkit2007/VOC2007/JPEGImages/$$n.jpg /content/demo_imgs/; done

!python demo.py --net res101 --cfg cfgs/res101.yml --dataset pascal_voc --cuda \
    --load_dir {SAVE_DIR} --image_dir /content/demo_imgs \
    --checksession 1 --checkepoch {CKPT_EPOCH} --checkpoint 11029

In [ ]:
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
files = sorted(glob.glob('/content/demo_imgs/*_det.jpg'))
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, f in zip(axes.flat, files):
    ax.imshow(mpimg.imread(f)); ax.axis('off')
plt.show()

## 6. Tracking (Kalman filter) pada satu sequence

`test_net.py` menjalankan detektor + Kalman filter pada sequence frame dan menghitung AP.

**Penting:** script ini **menimpa tiap frame dengan hasil crop-nya**, jadi harus dijalankan di salinan data, dan salinan itu harus dibuat ulang tiap kali dijalankan. Sel di bawah melakukannya. Box bola frame pertama dibaca dari anotasinya.

Sequence tersedia: `Put_1..Put_5`, `test_1`, `test_2`, `test_4`, `test_5`, `test_8`, `test_10`, `test_16`, `test_17`.

In [ ]:
SEQ = "Put_1"

import os, shutil
T = f"{DATA}/Tracking"
V = "/content/golf_ball/data/VOCdevkit2007/VOC2007"
names = [l.strip() for l in open(f"{T}/ImageSets/Main/{SEQ}.txt")]

# cari folder frame untuk sequence ini
src_dir = {n: os.path.join(r, n + ".jpg") for r, _, fs in os.walk(f"{T}/JPEGImages") for n in [f[:-4] for f in fs if f.endswith(".jpg")]}

if os.path.islink(V): os.unlink(V)
else: shutil.rmtree(V, ignore_errors=True)
shutil.rmtree("/content/golf_ball/data/cache", ignore_errors=True)
for d in ("Annotations", "JPEGImages", "ImageSets/Main"):
    os.makedirs(f"{V}/{d}")
for n in names:
    shutil.copy(src_dir[n], f"{V}/JPEGImages/{n}.jpg")
    shutil.copy(f"{T}/Annotations/{n}.xml", f"{V}/Annotations/{n}.xml")
shutil.copy(f"{T}/ImageSets/Main/{SEQ}.txt", f"{V}/ImageSets/Main/test.txt")
print(len(names), "frame disalin")

In [ ]:
!python test_net.py --dataset pascal_voc --net res101 --cfg cfgs/res101.yml --cuda \
    --load_dir {SAVE_DIR} --checksession 1 --checkepoch {CKPT_EPOCH} --checkpoint 11029

Untuk kembali ke dataset Detection (training/demo), jalankan ulang sel `ln -s` di bagian 2.